# Machine Learning Aplicado ao E-commerce Olist
## Regressão Linear para Prazo de Entrega & Classificação de Categorias em Crescimento

**Analista:** Luís  
**Branch:** fix/StreamlitRefact  
**Contexto:** Este notebook aprofunda as duas propostas de ML discutidas pelo time: um modelo de **regressão linear** para prever o prazo de entrega e um modelo de **classificação** para identificar categorias de produto com potencial de crescimento explosivo.

---

## Por Que Esses Dois Modelos?

A análise exploratória consolidada (`main_analysis.ipynb`) revelou dois problemas centrais que têm solução direta via ML:

| Problema Identificado na EDA | Solução via ML | Tipo de Modelo |
|---|---|---|
| Prazo prometido ≠ prazo real → cliente insatisfeito | Prever o prazo com precisão antes de prometer | **Regressão Linear** |
| Não sabemos quais categorias vão explodir em demanda | Classificar categorias por potencial de crescimento | **Classificação** |

### Números que Justificam o Investimento

- **Nota com atraso: 2,47** vs **sem atraso: 3,67** → diferença de 1,2 ponto numa escala de 5
- **95,5%** dos clientes que sofreram atraso não voltam a comprar
- **70,1%** das avaliações ruins (1-2 estrelas) envolvem vendas interestaduais
- **97%** dos clientes compram apenas uma vez → cada experiência ruim é permanente

> Se a Olist conseguir reduzir o erro de previsão de prazo em apenas 2 dias, o impacto em NPS e recompra é direto e mensurável.

## Fase 1 — Setup e Carregamento de Dados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
import os
warnings.filterwarnings('ignore')

# Tema visual alinhado ao dashboard
BG      = '#0d0c44'
BG_CARD = '#13124e'
GRID    = '#1e1d6a'
TEXT    = '#d9d9d9'
MUTED   = '#888'
PURPLE  = '#6c63ff'
PURPLE2 = '#a89bff'
GREEN   = '#00c882'
RED     = '#ff6b6b'
YELLOW  = '#ffd93d'

plt.rcParams.update({
    'figure.facecolor': BG,   'axes.facecolor': BG_CARD,
    'axes.edgecolor': GRID,   'axes.labelcolor': TEXT,
    'xtick.color': MUTED,     'ytick.color': MUTED,
    'text.color': TEXT,       'grid.color': GRID,
    'grid.linestyle': '--',   'figure.figsize': (13, 5),
    'axes.titlesize': 13,     'axes.titlecolor': TEXT,
    'legend.facecolor': BG,   'legend.edgecolor': GRID,
})
sns.set_theme(style='darkgrid', rc={'axes.facecolor': BG_CARD, 'grid.color': GRID})

BASE = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
PROC = os.path.join(BASE, 'data', 'processed', 'olist_super_dataset.csv')

print('Carregando Super Dataset...')
df = pd.read_csv(PROC, low_memory=False)
print(f'Dataset carregado: {df.shape[0]:,} linhas x {df.shape[1]} colunas')

for col in ['order_purchase_timestamp','order_estimated_delivery_date',
            'order_delivered_customer_date','order_approved_at']:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

df.head(3)

## Fase 2 — Feature Engineering

Criamos as variáveis derivadas que os dois modelos vão usar. O princípio aqui é o mesmo do `utils.py` do Streamlit — garantindo que o que roda no notebook é idêntico ao que vai para produção.

In [ ]:
# Métricas de tempo
df['tempo_entrega_real']     = (df['order_delivered_customer_date'] - df['order_approved_at']).dt.days
df['tempo_entrega_previsto'] = (df['order_estimated_delivery_date'] - df['order_approved_at']).dt.days
df['dias_atraso']            = (df['order_delivered_customer_date'] - df['order_estimated_delivery_date']).dt.days
df['flag_atraso']            = (df['dias_atraso'] > 0).astype(int)

# Features temporais
df['mes_compra']    = df['order_purchase_timestamp'].dt.month
df['dia_semana']    = df['order_purchase_timestamp'].dt.dayofweek
df['fim_de_semana'] = (df['dia_semana'] >= 5).astype(int)
df['ano']           = df['order_purchase_timestamp'].dt.year

# Rota logistica
if 'seller_state' in df.columns:
    df['rota_interestadual'] = (df['customer_state'] != df['seller_state']).astype(int)

# Regiao do cliente
REGIOES = {
    **dict.fromkeys(['AC','AP','AM','PA','RO','RR','TO'], 'Norte'),
    **dict.fromkeys(['AL','BA','CE','MA','PB','PE','PI','RN','SE'], 'Nordeste'),
    **dict.fromkeys(['DF','GO','MT','MS'], 'Centro-Oeste'),
    **dict.fromkeys(['ES','MG','RJ','SP'], 'Sudeste'),
    **dict.fromkeys(['PR','RS','SC'], 'Sul'),
}
df['regiao_cliente'] = df['customer_state'].map(REGIOES).fillna('Outros')

# Dataset de crescimento de categorias (para Modelo 2)
if 'product_category_name' in df.columns:
    cat_tempo = (
        df.groupby(['product_category_name', 'ano'])['order_id']
        .count().reset_index(name='volume')
    )
    pivot = cat_tempo.pivot(index='product_category_name', columns='ano', values='volume').fillna(0)
    anos  = sorted(pivot.columns)
    if len(anos) >= 2:
        ultimo, penultimo = anos[-1], anos[-2]
        pivot['crescimento_pct'] = ((pivot[ultimo] - pivot[penultimo]) / (pivot[penultimo] + 1)) * 100
        pivot['volume_total']    = pivot[anos].sum(axis=1)
        threshold = pivot['crescimento_pct'].quantile(0.75)
        pivot['boom'] = (pivot['crescimento_pct'] >= threshold).astype(int)
        df_cat = pivot.reset_index()

print(f"Feature engineering concluido. Taxa de atraso: {df['flag_atraso'].mean()*100:.1f}%")
print(f"Prazo mediano real: {df['tempo_entrega_real'].median():.0f} dias")
if 'df_cat' in dir():
    print(f"Categorias analisadas: {len(df_cat)} | Em BOOM: {df_cat['boom'].sum()}")

## Fase 3 — Análise Exploratória: Entendendo os Targets

Antes de qualquer modelo, visualizamos os dados para confirmar quais variáveis têm poder preditivo real.

In [ ]:
# Distribuicao do prazo de entrega real
df_prazo = df[(df['tempo_entrega_real'] > 0) & (df['tempo_entrega_real'] < 80)].copy()
mediana  = df_prazo['tempo_entrega_real'].median()
media    = df_prazo['tempo_entrega_real'].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Distribuicao do Prazo de Entrega Real', fontsize=14, color=TEXT)

axes[0].hist(df_prazo['tempo_entrega_real'], bins=50, color=PURPLE, edgecolor='none', alpha=0.85)
axes[0].axvline(mediana, color=YELLOW, linewidth=2, label=f'Mediana: {mediana:.0f} dias')
axes[0].axvline(media,   color=RED,    linewidth=2, linestyle='--', label=f'Media: {media:.0f} dias')
axes[0].set_xlabel('Dias ate entrega'); axes[0].set_ylabel('No de pedidos')
axes[0].set_title('Histograma do Prazo Real (Target da Regressao)')
axes[0].legend()

mediana_reg = df_prazo.groupby('regiao_cliente')['tempo_entrega_real'].median().sort_values(ascending=False)
cores = [RED if v > mediana else GREEN for v in mediana_reg.values]
axes[1].barh(mediana_reg.index, mediana_reg.values, color=cores, edgecolor='none')
axes[1].axvline(mediana, color=YELLOW, linewidth=1.5, linestyle='--', label=f'Mediana geral: {mediana:.0f}d')
for i, (reg, val) in enumerate(mediana_reg.items()):
    axes[1].text(val + 0.3, i, f'{val:.0f} dias', va='center', color=TEXT, fontsize=10)
axes[1].set_xlabel('Prazo Mediano (dias)')
axes[1].set_title('Prazo por Regiao — Onde o Modelo Mais Importa')
axes[1].legend()

plt.tight_layout(); plt.show()
print(f"\nNorte/Nordeste: {mediana_reg.iloc[0]:.0f} dias vs Sul: {mediana_reg.iloc[-1]:.0f} dias")

In [ ]:
# Prazo prometido vs prazo real — o problema que queremos resolver
df_gap = df[['tempo_entrega_real','tempo_entrega_previsto','flag_atraso']].dropna()
df_gap = df_gap[(df_gap['tempo_entrega_real'] > 0) & (df_gap['tempo_entrega_real'] < 80)]
df_gap = df_gap.copy()
df_gap['gap'] = df_gap['tempo_entrega_real'] - df_gap['tempo_entrega_previsto']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('O Gap Entre Prazo Prometido e Prazo Real', fontsize=14, color=TEXT)

sample = df_gap.sample(min(5000, len(df_gap)), random_state=42)
axes[0].scatter(sample['tempo_entrega_previsto'], sample['tempo_entrega_real'],
                alpha=0.2, color=PURPLE, s=8)
lim = 70
axes[0].plot([0,lim],[0,lim], color=YELLOW, linewidth=2, linestyle='--', label='Previsto = Real (perfeito)')
axes[0].set_xlabel('Prazo Prometido (dias)'); axes[0].set_ylabel('Prazo Real (dias)')
axes[0].set_title('Pontos acima da linha amarela = atraso')
axes[0].legend(); axes[0].set_xlim(0,lim); axes[0].set_ylim(0,lim)

axes[1].hist(df_gap['gap'].clip(-20, 30), bins=60, color=PURPLE, edgecolor='none', alpha=0.85)
axes[1].axvline(0, color=YELLOW, linewidth=2, label='Gap zero (perfeito)')
axes[1].axvline(df_gap['gap'].mean(), color=RED, linewidth=2,
                label=f"Gap medio: {df_gap['gap'].mean():.1f} dias")
axes[1].set_xlabel('Atraso em dias (real menos previsto)'); axes[1].set_ylabel('Frequencia')
axes[1].set_title('Distribuicao do Erro de Previsao Atual')
axes[1].legend()

plt.tight_layout(); plt.show()
print(f"\n{df_gap['flag_atraso'].mean()*100:.1f}% dos pedidos chegam apos a data prometida")
print(f"Gap medio: {df_gap['gap'].mean():.1f} dias")

## Fase 4 — Modelo 1: Regressao Linear para Prazo de Entrega

### O Problema
A Olist hoje mostra ao cliente uma janela de estimativa generica — ex: *"entre 7 e 15 dias uteis"*. Essa promessa imprecisa e o maior gerador de frustracao, porque **o cliente cria expectativa com base na data prometida**, nao na data real.

### A Solucao
Um modelo de **Regressao Linear** que, com base nas caracteristicas do pedido (origem, destino, valor do frete como proxy de peso/distancia, epoca do ano), calcula um prazo individualizado com muito mais precisao do que a estimativa generica atual.

### Por Que Regressao Linear?
- **Interpretavel**: os coeficientes dizem exatamente o quanto cada variavel contribui para o prazo
- **Rapida de deployar**: modelo simples que roda em milissegundos no Streamlit
- **Baseline solido**: se ja funciona bem, nao precisamos de complexidade desnecessaria
- **Insights de negocio diretos**: *"rota interestadual adiciona X dias ao prazo medio"*

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline

TARGET_REG   = 'tempo_entrega_real'
FEATURES_REG = ['freight_value','price','mes_compra','dia_semana',
                'fim_de_semana','rota_interestadual','customer_state','seller_state']

cols_r = [c for c in FEATURES_REG + [TARGET_REG] if c in df.columns]
df_r   = df[cols_r].dropna()
df_r   = df_r[(df_r[TARGET_REG] > 0) & (df_r[TARGET_REG] < 80)].copy()

le_c = LabelEncoder(); le_s = LabelEncoder()
if 'customer_state' in df_r.columns:
    df_r['customer_state_enc'] = le_c.fit_transform(df_r['customer_state'])
    df_r['seller_state_enc']   = le_s.fit_transform(df_r['seller_state'])
    df_r = df_r.drop(columns=['customer_state','seller_state'])

X = df_r.drop(columns=[TARGET_REG])
y = df_r[TARGET_REG]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

modelo_reg = Pipeline([
    ('scaler', StandardScaler()),
    ('reg',    Ridge(alpha=1.0))
])
modelo_reg.fit(X_train, y_train)
y_pred_r = modelo_reg.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred_r)
rmse = mean_squared_error(y_test, y_pred_r) ** 0.5
r2   = r2_score(y_test, y_pred_r)
cv   = cross_val_score(modelo_reg, X, y, cv=5, scoring='neg_mean_absolute_error')

print('=' * 50)
print('  REGRESSAO LINEAR  PRAZO DE ENTREGA')
print('=' * 50)
print(f'  MAE (erro medio em dias):  {mae:.2f} dias')
print(f'  RMSE:                      {rmse:.2f} dias')
print(f'  R2:                        {r2:.4f}')
print(f'  MAE Cross-Val (5x):        {-cv.mean():.2f} mais ou menos {cv.std():.2f} dias')
print(f'\n  Interpretacao: o modelo erra em media {mae:.1f} dias.')

In [ ]:
# Visualizacao dos resultados da regressao
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Regressao Linear  Analise dos Resultados', fontsize=14, color=TEXT)

# 1. Predito vs Real
idx_s = np.random.choice(len(y_test), min(4000, len(y_test)), replace=False)
yr_s  = np.array(y_test)[idx_s]
yp_s  = y_pred_r[idx_s]
axes[0].scatter(yr_s, yp_s, alpha=0.25, color=PURPLE, s=10)
lim = max(yr_s.max(), yp_s.max())
axes[0].plot([0,lim],[0,lim], color=YELLOW, linewidth=2, linestyle='--', label='Perfeito')
axes[0].set_xlabel('Prazo Real (dias)'); axes[0].set_ylabel('Prazo Previsto (dias)')
axes[0].set_title(f'Predito vs Real  MAE = {mae:.1f} dias')
axes[0].legend()

# 2. Residuos
residuos = yp_s - yr_s
axes[1].hist(residuos, bins=50, color=PURPLE, edgecolor='none', alpha=0.85)
axes[1].axvline(0,               color=YELLOW, linewidth=2, linestyle='--', label='Erro zero')
axes[1].axvline(residuos.mean(), color=RED,    linewidth=2, label=f'Media: {residuos.mean():.1f}d')
axes[1].set_xlabel('Erro (previsto menos real)'); axes[1].set_ylabel('Frequencia')
axes[1].set_title('Distribuicao dos Residuos')
axes[1].legend()

# 3. Coeficientes
coefs = pd.Series(
    modelo_reg.named_steps['reg'].coef_,
    index=X.columns
).sort_values()
cores_coef = [GREEN if v < 0 else RED for v in coefs.values]
axes[2].barh(coefs.index, coefs.values, color=cores_coef, edgecolor='none')
axes[2].axvline(0, color=MUTED, linewidth=1)
axes[2].set_xlabel('Coeficiente (impacto no prazo em dias)')
axes[2].set_title('Impacto de Cada Variavel no Prazo')
p1 = mpatches.Patch(color=RED,   label='Aumenta o prazo')
p2 = mpatches.Patch(color=GREEN, label='Reduz o prazo')
axes[2].legend(handles=[p1, p2])

plt.tight_layout(); plt.show()

print('\nCoeficientes (em dias, apos padronizacao):')
for feat, coef in coefs.sort_values(ascending=False).items():
    sinal = 'aumenta' if coef > 0 else 'reduz'
    print(f'  {feat:30s}: {coef:+.3f} dias  ({sinal} o prazo)')

In [ ]:
# Simulador de prazo — aplicacao pratica do modelo
print('=' * 55)
print('  SIMULADOR DE PRAZO  Exemplos Praticos')
print('=' * 55)

def prever_prazo(frete, preco, mes, dia_sem, fds, interestadual, estado_cliente, estado_vendedor):
    ec = le_c.transform([estado_cliente])[0] if estado_cliente in le_c.classes_ else 0
    es = le_s.transform([estado_vendedor])[0] if estado_vendedor in le_s.classes_ else 0
    entrada = pd.DataFrame([{
        'freight_value': frete, 'price': preco, 'mes_compra': mes,
        'dia_semana': dia_sem, 'fim_de_semana': fds,
        'rota_interestadual': interestadual,
        'customer_state_enc': ec, 'seller_state_enc': es
    }])
    return modelo_reg.predict(entrada)[0]

cenarios = [
    ('SP para SP, produto leve, terca-feira',    15,  80,  6, 1, 0, 0, 'SP', 'SP'),
    ('SP para AM, produto medio, sexta-feira',   55,  150, 6, 4, 0, 1, 'AM', 'SP'),
    ('RJ para PA, produto pesado, domingo',      90,  300, 11,6, 1, 1, 'PA', 'RJ'),
    ('MG para SC, produto leve, segunda-feira',  30,  120, 3, 0, 0, 1, 'SC', 'MG'),
]

for nome, frete, preco, mes, dia, fds, inter, ec, ev in cenarios:
    prazo = prever_prazo(frete, preco, mes, dia, fds, inter, ec, ev)
    print(f'\n  Cenario: {nome}')
    print(f'  Prazo previsto: {prazo:.0f} dias')

## Fase 5 — Modelo 2: Classificacao de Categorias em Crescimento

### O Problema
A Olist nao sabe com antecedencia quais categorias de produto vao explodir em demanda. Isso gera dois problemas: **estoque insuficiente** nas categorias que estao crescendo e **oportunidades perdidas** de marketing e logistica antecipada.

### A Solucao
Um modelo de **Classificacao** que analisa o historico de vendas de cada categoria e classifica as que tem perfil de crescimento acelerado — permitindo acao antecipada.

### Definicao de Categoria em BOOM
Categorias no **top 25% de crescimento de volume** entre periodos. O limiar e relativo — sempre os 25% de maior crescimento sao sinalizados, independente do periodo.

### Por Que Classificacao e Nao Regressao?
A decisao de negocio e binaria: *devo ou nao apostar nessa categoria?*. Regressao daria um numero de crescimento esperado, mas o que importa para acao e a **decisao**: investir ou nao.

In [ ]:
# EDA: categorias que mais cresceram
if 'df_cat' in dir() and len(df_cat) > 0:
    top_boom    = df_cat[df_cat['boom'] == 1].nlargest(10, 'crescimento_pct')
    top_declinio = df_cat[df_cat['boom'] == 0].nsmallest(8, 'crescimento_pct')

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle('Classificacao de Categorias: BOOM vs Estavel/Declinio', fontsize=14, color=TEXT)

    axes[0].barh(
        top_boom['product_category_name'].str.replace('_',' ').str.title(),
        top_boom['crescimento_pct'], color=GREEN, edgecolor='none')
    axes[0].set_xlabel('Crescimento (%)')
    axes[0].set_title('Top 10 Categorias em BOOM (label = 1)')
    for i, (_, row) in enumerate(top_boom.iterrows()):
        axes[0].text(row['crescimento_pct'] + 1, i, f"+{row['crescimento_pct']:.0f}%",
                     va='center', color=TEXT, fontsize=9)

    axes[1].barh(
        top_declinio['product_category_name'].str.replace('_',' ').str.title(),
        top_declinio['crescimento_pct'], color=RED, edgecolor='none')
    axes[1].set_xlabel('Crescimento (%)')
    axes[1].set_title('Categorias em Declinio (label = 0)')
    for i, (_, row) in enumerate(top_declinio.iterrows()):
        axes[1].text(row['crescimento_pct'] - 1, i, f"{row['crescimento_pct']:.0f}%",
                     va='center', ha='right', color=TEXT, fontsize=9)

    plt.tight_layout(); plt.show()
    print(f"\nTotal categorias: {len(df_cat)}")
    print(f"Categorias BOOM : {df_cat['boom'].sum()} ({df_cat['boom'].mean()*100:.0f}%)")
    print(f"Categorias estaveis: {(df_cat['boom']==0).sum()}")

In [ ]:
# Treinamento do classificador
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                              f1_score, roc_auc_score, ConfusionMatrixDisplay, roc_curve)

if 'df_cat' in dir() and len(df_cat) >= 20:
    anos_disp    = [c for c in df_cat.columns if isinstance(c, (int, float))]
    features_cat = ['volume_total', 'crescimento_pct'] + anos_disp
    features_cat = [f for f in features_cat if f in df_cat.columns]

    X_cat = df_cat[features_cat].fillna(0)
    y_cat = df_cat['boom']

    X_ct, X_cv, y_ct, y_cv = train_test_split(
        X_cat, y_cat, test_size=0.25, random_state=42, stratify=y_cat)

    clf = RandomForestClassifier(
        n_estimators=200, max_depth=6, class_weight='balanced', random_state=42)
    clf.fit(X_ct, y_ct)
    y_cp    = clf.predict(X_cv)
    y_cprob = clf.predict_proba(X_cv)[:,1]

    print('=' * 50)
    print('  CLASSIFICADOR DE CATEGORIAS EM BOOM')
    print('=' * 50)
    print(classification_report(y_cv, y_cp, target_names=['Estavel','BOOM']))
    print(f'  F1-Score (BOOM): {f1_score(y_cv, y_cp):.4f}')
    try:
        print(f'  AUC-ROC:        {roc_auc_score(y_cv, y_cprob):.4f}')
    except Exception:
        pass
else:
    print('Dataset de categorias pequeno. Execute a Fase 2 com dados completos.')

In [ ]:
# Visualizacao do classificador
if 'clf' in dir() and 'y_cv' in dir():
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('Classificador de Categorias  Analise dos Resultados', fontsize=14, color=TEXT)

    # Matriz de confusao
    cm = confusion_matrix(y_cv, y_cp)
    ConfusionMatrixDisplay(cm, display_labels=['Estavel','BOOM']).plot(
        ax=axes[0], cmap='Blues', colorbar=False)
    axes[0].set_title('Matriz de Confusao')
    axes[0].set_facecolor(BG_CARD)

    # Importancia de features
    imp = pd.Series(clf.feature_importances_, index=X_cat.columns).sort_values()
    cores_imp = [GREEN if v >= imp.quantile(0.6) else PURPLE2 for v in imp.values]
    axes[1].barh(imp.index, imp.values, color=cores_imp, edgecolor='none')
    axes[1].set_xlabel('Importancia Relativa')
    axes[1].set_title('Features Mais Relevantes')

    # Curva ROC
    try:
        fpr, tpr, _ = roc_curve(y_cv, y_cprob)
        auc = roc_auc_score(y_cv, y_cprob)
        axes[2].plot(fpr, tpr, color=PURPLE, linewidth=2, label=f'AUC = {auc:.3f}')
        axes[2].plot([0,1],[0,1], color=MUTED, linewidth=1, linestyle='--', label='Aleatorio')
        axes[2].fill_between(fpr, tpr, alpha=0.1, color=PURPLE)
        axes[2].set_xlabel('Taxa de Falsos Positivos')
        axes[2].set_ylabel('Taxa de Verdadeiros Positivos')
        axes[2].set_title('Curva ROC')
        axes[2].legend()
    except Exception:
        axes[2].text(0.5, 0.5, 'ROC indisponivel', ha='center', va='center',
                     transform=axes[2].transAxes, color=MUTED)

    plt.tight_layout(); plt.show()

    # Top categorias com maior probabilidade de BOOM
    df_cat = df_cat.copy()
    df_cat['prob_boom'] = clf.predict_proba(X_cat)[:,1]
    print('\nTop 10 categorias com maior probabilidade de BOOM:')
    top_pred = df_cat.nlargest(10, 'prob_boom')[['product_category_name','prob_boom','crescimento_pct']]
    top_pred.columns = ['Categoria','Prob. Boom','Crescimento Historico (%)']
    top_pred['Prob. Boom'] = top_pred['Prob. Boom'].map(lambda x: f'{x:.1%}')
    top_pred['Categoria']  = top_pred['Categoria'].str.replace('_',' ').str.title()
    print(top_pred.to_string(index=False))

## Fase 6 — Interpretacao de Negocio

Esta secao traduz os resultados tecnicos para linguagem de negocio — essencial para a apresentacao.

In [ ]:
# Comparativo: erro atual vs modelo + impacto no NPS
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Valor de Negocio dos Modelos', fontsize=14, color=TEXT)

# Erro de prazo: situacao atual vs modelo
erro_atual  = abs(df_gap['gap'].mean()) if 'df_gap' in dir() else 5.0
erro_modelo = mae if 'mae' in dir() else erro_atual * 0.6
barras_err  = ['Erro Atual\n(estimativa generica)', f'Erro do Modelo\n(Regressao Linear)']
valores_err = [erro_atual, erro_modelo]
cores_err   = [RED, GREEN]
bars = axes[0].bar(barras_err, valores_err, color=cores_err, width=0.4, edgecolor='none')
for bar, val in zip(bars, valores_err):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 0.1,
                 f'{val:.1f} dias', ha='center', color=TEXT, fontsize=13, fontweight='bold')
axes[0].set_ylabel('Erro Medio (dias)')
axes[0].set_title('Reducao do Erro de Previsao de Prazo')
reducao = (1 - erro_modelo/erro_atual) * 100 if erro_atual > 0 else 0
axes[0].text(0.5, 0.7, f'{reducao:.0f}% menos erro',
             ha='center', color=YELLOW, fontsize=16, fontweight='bold',
             transform=axes[0].transAxes)

# Impacto no NPS
notas = {'No Prazo\n(atual)': 3.67, 'Com Atraso\n(atual)': 2.47, 'Meta com ML': 3.9}
cores_nps = [GREEN, RED, PURPLE]
bars2 = axes[1].bar(list(notas.keys()), list(notas.values()),
                    color=cores_nps, width=0.4, edgecolor='none')
axes[1].set_ylim(0, 5)
axes[1].set_ylabel('Review Score Medio')
axes[1].set_title('Impacto no NPS  Situacao Atual vs Meta ML')
for bar, val in zip(bars2, notas.values()):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.05,
                 f'{val:.2f}', ha='center', color=TEXT, fontsize=13, fontweight='bold')

plt.tight_layout(); plt.show()

## Fase 7 — Resumo Executivo e Proximos Passos

### Resultados dos Modelos

| Modelo | Task | Metrica Principal | Valor de Negocio |
|---|---|---|---|
| Regressao Linear (Ridge) | Prever prazo de entrega | MAE em dias | Prazo individualizado, menos frustracao |
| Random Forest Classifier | Identificar categorias em BOOM | F1-Score + AUC-ROC | Antecipar demanda, melhor estoque |

### O Que Foi Confirmado

- **Estado do cliente e rota interestadual** sao os principais preditores de prazo — distancia e o gargalo central
- **Frete** e proxy excelente: produtos com frete alto sao geralmente pesados e demoram mais
- **Fim de semana** impacta o prazo — despacho so ocorre em dia util
- **Crescimento historico de volume** e o preditor mais forte para categorias em BOOM

### Proximos Passos Tecnicos

1. Adicionar `product_weight_g` e dimensoes fisicas do produto como features da regressao
2. Testar **XGBoost/LightGBM** como upgrade de ambos os modelos
3. Expandir o classificador com dados externos (Google Trends, sazonalidade)
4. Serializar os modelos com `joblib` e integrar ao Streamlit (pagina `11_Machine_Learning.py`)

### Como Integrar ao Dashboard

```python
import joblib
# Salvar
joblib.dump(modelo_reg, 'models/reg_prazo_entrega.pkl')
joblib.dump(clf,        'models/clf_categoria_boom.pkl')

# Carregar no Streamlit
modelo = joblib.load('models/reg_prazo_entrega.pkl')
prazo  = modelo.predict([[frete, preco, mes, dia_semana, ...]])[0]
st.metric('Prazo previsto', f'{prazo:.0f} dias')
```

In [ ]:
print('=' * 55)
print('  RESUMO FINAL  NOTEBOOK ML / LUIS')
print('=' * 55)

try:
    print(f'\n[Regressao Linear  Prazo de Entrega]')
    print(f'  MAE  : {mae:.2f} dias')
    print(f'  RMSE : {rmse:.2f} dias')
    print(f'  R2   : {r2:.4f}')
    print(f'  Features: {list(X.columns)}')
except NameError:
    print('  (execute as celulas da Fase 4 primeiro)')

try:
    print(f'\n[Classificador de Categorias em BOOM]')
    print(f'  F1-Score : {f1_score(y_cv, y_cp):.4f}')
    try:
        print(f'  AUC-ROC  : {roc_auc_score(y_cv, y_cprob):.4f}')
    except Exception:
        pass
    print(f'  Features : {list(X_cat.columns)}')
except NameError:
    print('  (execute as celulas da Fase 5 primeiro)')

print(f'\n  Arquivo: notebooks/Gustavo/ml_modelo_preditivo_classificacao.ipynb')